# Day 5: Query Layer — `load_bonds_for_date`\n\nTests for `src/termstructure/io.py`. Verifies that DuckDB can query the parquet on disk and return correct results for normal dates, weekends, and edge cases.

In [ ]:
import sys
sys.path.insert(0, "../src")

from termstructure.io import load_bonds_for_date

## 1. Normal trading day\n\nShould return exactly 1 row with 37 columns.

In [ ]:
df = load_bonds_for_date('2020-01-15')
print('Shape:', df.shape)
assert df.shape[0] == 1, 'Expected 1 row'
assert df.shape[1] == 37, 'Expected 37 columns'
df

## 2. Weekend date\n\nJan 11 2020 is a Saturday — no Fed data exists. Should return an empty DataFrame, not an error.

In [ ]:
df_weekend = load_bonds_for_date('2020-01-11')
print('Shape:', df_weekend.shape)
assert df_weekend.shape[0] == 0, 'Expected 0 rows for a weekend'
print('OK — empty DataFrame as expected')

## 3. Yield curve sanity check\n\nFor any real date, the yield curve should slope upward on average — short-term rates below long-term rates. We extract the 1Y, 5Y, 10Y, and 30Y zero yields and check the ordering.

In [ ]:
df = load_bonds_for_date('2020-01-15')
y1  = df['sveny01'].iloc[0]
y5  = df['sveny05'].iloc[0]
y10 = df['sveny10'].iloc[0]
y30 = df['sveny30'].iloc[0]

print(f'1Y:  {y1:.4f}%')
print(f'5Y:  {y5:.4f}%')
print(f'10Y: {y10:.4f}%')
print(f'30Y: {y30:.4f}%')
assert y1 < y30, 'Expected 1Y yield below 30Y yield (normal curve)'
print('OK — curve slopes upward')

## 4. Inverted curve — 2023\n\nThe 2022-2023 rate hiking cycle produced an inverted curve where short rates exceeded long rates. This checks that the data captures it correctly (1Y > 10Y expected).

In [ ]:
df_inv = load_bonds_for_date('2023-07-03')
y1  = df_inv['sveny01'].iloc[0]
y10 = df_inv['sveny10'].iloc[0]

print(f'1Y:  {y1:.4f}%')
print(f'10Y: {y10:.4f}%')
print(f'Spread (10Y-1Y): {y10 - y1:.4f}%')
assert y1 > y10, 'Expected inverted curve in mid-2023'
print('OK — inverted curve confirmed')

## 5. Plot the yield curve for one date

In [ ]:
import matplotlib.pyplot as plt

df = load_bonds_for_date('2020-01-15')
maturities = list(range(1, 31))
yields = [df[f'sveny{m:02d}'].iloc[0] for m in maturities]

plt.figure(figsize=(10, 4))
plt.plot(maturities, yields, marker='o', markersize=3)
plt.xlabel('Maturity (years)')
plt.ylabel('Zero yield (%)')
plt.title('Treasury Zero Curve — 2020-01-15')
plt.grid(True)
plt.tight_layout()
plt.show()